# ⚡ Suha AGI Studio — 100% Free GPU Inference Bridge
Jambatan inferens Google Colab GPU (Tesla T4 16GB VRAM) untuk model Uncensored dari Hugging Face ke 9Router VPS.

In [ ]:
# 1. Pasang enjin inferens llama-cpp-python & Cloudflare Tunnel
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python uvicorn fastapi huggingface_hub -q
!curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -o cloudflared.deb
!dpkg -i cloudflared.deb > /dev/null
print('✓ Enjin CUDA & Cloudflare Tunnel Siap Dipasang!')

In [ ]:
# 2. Muat turun model Qwen Instruct GGUF daripada Hugging Face
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id='bartowski/Qwen2.5-7B-Instruct-1M-GGUF',
    filename='Qwen2.5-7B-Instruct-1M-Q4_K_M.gguf'
)
print(f'✓ Model Loaded at: {model_path}')

In [ ]:
# 3. Lancarkan Pelayan API GPU (Port 8000) & Buka Terowong Serentak
import subprocess, time, urllib.request, re

# Jalankan server di latar belakang
server_cmd = f'python3 -m llama_cpp.server --model {model_path} --n_gpu_layers 35 --n_ctx 4096 --port 8000 --host 127.0.0.1'
server_proc = subprocess.Popen(server_cmd, shell=True)

# Tunggu sehingga pelayan Port 8000 benar-benar bersedia
print('Sedang memuatkan model ke VRAM GPU T4... Sila tunggu sebentar.')
server_ready = False
for _ in range(40):
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=2) as r:
            if r.status == 200:
                server_ready = True
                break
    except:
        time.sleep(2)

if server_ready:
    print('🟢 Pelayan GPU Berjaya Hidup di Port 8000!')
    # Buka Cloudflare Tunnel
    tunnel_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True
    )
    
    for line in iter(tunnel_proc.stdout.readline, ''):
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print('='*60)
            print(f'⚡ LIVE INFERENCE ENDPOINT: {match.group(0)}')
            print('='*60)
            break
else:
    print('❌ Pelayan mengambil masa terlalu lama untuk hidup. Sila jalankan sel ini semula.')